In [ ]:
import requests
import pandas as pd
import time
import os
import glob
import re
import wikipedia
from bs4 import BeautifulSoup, Tag
# from bs4.element import Tag
from datetime import datetime

from tqdm.auto import tqdm
tqdm.pandas()

from dotenv import load_dotenv

In [ ]:
load_dotenv(dotenv_path="/Users/apple/Code/utmtenv")
WATCHMODE_API_KEY = os.getenv("WATCHMODE_API_KEY", "")
REGION = 'IN'  # India
SOURCE = 'amazon_prime'

In [ ]:
def fetch_prime_movies(pages=100, limit=100):
    """Fetches a list of movies currently on Amazon Prime via Watchmode."""
    all_titles = []
    base_url = 'https://api.watchmode.com/v1/list-titles/'
    
    try:
        for page in range(1, pages + 1):
            print(f"📡 Fetching Watchmode Page {page}...")
            params = {
                'apiKey': WATCHMODE_API_KEY,
                'sources': SOURCE,
                'regions': REGION,
                'types': 'movie',
                'limit': limit,
                'page': page
            }
            response = requests.get(base_url, params=params)
            data = response.json()
            
            if 'titles' not in data or not data['titles']:
                break
                
            all_titles.extend(data['titles'])
            
            print(len(data['titles']), len(all_titles))
            time.sleep(1.0) # Rate limit protection
    except Exception as e:
        print(str(e))
        
    return pd.DataFrame(all_titles)

In [ ]:
# 1. Get Base Catalog
print("🚀 Starting Data Extraction...")
df = fetch_prime_movies() # Adjust limit as needed
print(f"✅ Found {len(df)} movies on Prime.")
df

In [ ]:
df = df[df["Language"].isin(["English", "Hindi"])].reset_index(drop=True)
df.rename(columns={"Movie Name": "title", "Year of Release": "year"}, inplace=True)
df

In [ ]:
# 1. Sort by the 'year' column in descending order (highest year first)
df_sorted = df.sort_values(by='year', ascending=False)

# 2. Drop duplicates based on 'title', keeping the very first one it sees
df_clean = df_sorted.drop_duplicates(subset=['title'], keep='first')

# (Optional) If you want to restore the original order of your rows after cleaning
df_clean = df_clean.sort_index()
df_clean

In [ ]:
def get_wikipedia_data(row):
    """Searches Wikipedia and extracts Plot, Cast, Box Office, Critical Response, and Infobox data."""
    
    title = row.get("title", "")
    year = row.get("year", "")
    
    info = {
        "plot_summary": None,
        "cast": None,
        "box_office": None,
        "critical_response": None,
        "director": None,
        "budget": None,
        "running_time": None,
        "all_infobox_data": {} 
    }
    
    try:
        search_query = f"{title} ({int(year)}) film" if pd.notna(year) and year else f"{title} film"
        search_query_result = wikipedia.search(search_query[:300])
        
        if not search_query_result:
            return pd.Series(info)
            
        page_title = search_query_result[0]
        page = wikipedia.page(page_title, auto_suggest=False)
        
        res = requests.get(page.url, headers={'User-Agent': 'Mozilla/5.0'})
        soup = BeautifulSoup(res.text, 'html.parser')

        # 1. Parse Infobox (Unchanged)
        infobox = soup.find('table', class_='infobox')
        if infobox:
            for tr in infobox.find_all('tr'):
                th = tr.find('th')
                td = tr.find('td')
                if th and td:
                    key = th.get_text(separator=" ", strip=True).replace('\xa0', ' ')
                    for sup in td.find_all(['sup', 'style']):
                        sup.decompose()
                    value = td.get_text(separator=", ", strip=True) 
                    info["all_infobox_data"][key] = value
                    
                    key_lower = key.lower()
                    if 'box office' in key_lower: info["box_office"] = value
                    elif 'starring' in key_lower: info["cast"] = value
                    elif 'directed by' in key_lower: info["director"] = value
                    elif 'budget' in key_lower: info["budget"] = value
                    elif 'running time' in key_lower: info["running_time"] = value

        # --- UPDATED HELPER FUNCTION ---
        def extract_section_text(section_keywords, limit_words=None):
            # Look for h2 or h3 headers directly
            for header in soup.find_all(['h2', 'h3']):
                # Get the text of the header, removing the "[edit]" button text
                header_text = header.get_text(strip=True).lower().replace('[edit]', '').strip()
                
                # Check if this header matches what we are looking for
                if any(kw in header_text for kw in section_keywords):
                    
                    # THE FIX: If Wikipedia wrapped the header in a div, step up to the div.
                    # Otherwise, use the header itself (for older page layouts).
                    if header.parent and 'mw-heading' in header.parent.get('class', []):
                        current_node = header.parent
                    else:
                        current_node = header

                    paragraphs = []
                    # Iterate through everything that comes AFTER the header/div
                    for elem in current_node.next_siblings:
                        if isinstance(elem, Tag):
                            # Stop grabbing text if we hit the next major section
                            if elem.name in ['h2', 'h3'] or 'mw-heading' in elem.get('class', []):
                                break
                            # Grab paragraph text
                            if elem.name == 'p':
                                text = elem.get_text(separator=' ', strip=True)
                                if text: paragraphs.append(text)
                                
                    raw_text = ' '.join(paragraphs)
                    clean_text = re.sub(r'\[\s*\d+\s*\]', '', raw_text).strip()
                    
                    if clean_text:
                        if limit_words:
                            words = clean_text.split()
                            return " ".join(words[:limit_words]) + ("..." if len(words) > limit_words else "")
                        return clean_text
            return None
        # -------------------------------

        # 2. Extract Plot (no limit)
        info["plot_summary"] = extract_section_text(['plot', 'premise', 'synopsis'])
        
        # 3. Extract Critical Response (limited to ~50 words)
        info["critical_response"] = extract_section_text(['critical response', 'reception', 'critical reception'], limit_words=50)

        # 4. Fallback for Cast
        if not info["cast"]:
            info["cast"] = extract_section_text(['cast'], limit_words=100)

    except Exception as e:
        print(f"Error fetching data for '{title}': {str(e)}")
        
    return pd.Series(info)
    
# wiki_data_df = df.progress_apply(get_wikipedia_data, axis=1)
# df_final = pd.concat([df, wiki_data_df], axis=1)
# df_final

In [ ]:
CHECKPOINT_FILE = "wikipedia_results_checkpoint.xlsx"
CHUNK_SIZE = 50  

# 1. Check what has already been processed
if os.path.exists(CHECKPOINT_FILE):
    # Load the completed rows from Excel
    completed_df = pd.read_excel(CHECKPOINT_FILE)
    completed_titles = completed_df['title'].tolist() 
    print(f"Found existing file. Resuming... ({len(completed_titles)} rows already processed)")
else:
    completed_titles = []
    print("Starting fresh...")

# 2. Filter your original DataFrame
df_remaining = df_clean[~df_clean['title'].isin(completed_titles)].copy()
print(f"Rows left to process: {len(df_remaining)}")

# 3. Process the remaining rows in chunks
for i in range(0, len(df_remaining), CHUNK_SIZE):
    chunk = df_remaining.iloc[i : i + CHUNK_SIZE].copy()
    
    print(f"\nProcessing rows {i} to {i + len(chunk)}...")
    
    # Run the web scraping function
    wiki_data_df = chunk.progress_apply(get_wikipedia_data, axis=1)
    
    # Merge back
    chunk_final = pd.concat([chunk, wiki_data_df], axis=1)
    
    # 4. Save to Excel safely
    if os.path.exists(CHECKPOINT_FILE):
        # If the file exists, we open it in append mode ('a')
        with pd.ExcelWriter(CHECKPOINT_FILE, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
            # Find the last row in the existing sheet so we don't overwrite data
            startrow = writer.sheets['Sheet1'].max_row
            
            # Write the new chunk starting at that row (without headers)
            chunk_final.to_excel(writer, sheet_name='Sheet1', startrow=startrow, index=False, header=False)
    else:
        # If it's the very first chunk, we create the file normally (with headers)
        chunk_final.to_excel(CHECKPOINT_FILE, sheet_name='Sheet1', index=False)

print("\nAll done! Data saved to", CHECKPOINT_FILE)

In [ ]:
df_new = pd.read_excel("wikipedia_results_checkpoint.xlsx")
df_new

In [ ]:
import ast

# 2. Define a helper function to safely parse the string
def parse_dict_string(val):
    """Safely converts a string representation of a dict to an actual dict."""
    # Return an empty dict if the value is missing (NaN)
    if pd.isna(val):
        return {}
    
    try:
        # literal_eval safely parses strings containing Python data structures
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        # Fallback for malformed strings
        return {}

# 3. Apply the parser to convert strings to dictionaries
parsed_series = df_new['all_infobox_data'].apply(parse_dict_string)

# 4. Convert the series of dictionaries directly into a new DataFrame
# This method is significantly faster than using .apply(pd.Series)
extracted_df = pd.DataFrame(parsed_series.tolist())

# 5. Concatenate the new columns with the original DataFrame
# We drop the original 'all_infobox_data' column to clean up


# Display the results
# print(final_df.head())

In [ ]:
final_df = pd.concat([df_new.drop(columns=['all_infobox_data']), extracted_df], axis=1)